In [2]:
# https://dacon.io/competitions/open/235576/overview/description
# 서울시 따릉이 대여량 예측 경진대회

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, root_mean_squared_error
import pandas as pd

In [3]:
# 1. 데이터
# path = "./_data/ddarung/"   # 상대경로 # 현재위치는 study 디렉토리임
path = "c:/study/_data/ddarung/"    # 절대경로
# path = "c:\study\_data\ddarung/"    # 슬래시, 역슬래시 상관없음
# path = "c://study//_data//ddarung/"
# path = "c:\\study\\_data\\ddarung\\"

train_csv = pd.read_csv(path + "train.csv", index_col=0)
display(train_csv.head())

test_csv = pd.read_csv(path + "test.csv", index_col=0)
display(test_csv.head())

submission = pd.read_csv(path + "submission.csv", index_col=0)
display(submission.head())

,hour,hour_bef_temperature,hour_bef_precipitation,hour_bef_windspeed,hour_bef_humidity,hour_bef_visibility,hour_bef_ozone,hour_bef_pm10,hour_bef_pm2.5,count
id,,,,,,,,,,
3,20,16.3,1.0,1.5,89.0,576.0,0.027,76.0,33.0,49.0
6,13,20.1,0.0,1.4,48.0,916.0,0.042,73.0,40.0,159.0
7,6,13.9,0.0,0.7,79.0,1382.0,0.033,32.0,19.0,26.0
8,23,8.1,0.0,2.7,54.0,946.0,0.040,75.0,64.0,57.0
9,18,29.5,0.0,4.8,7.0,2000.0,0.057,27.0,11.0,431.0


,hour,hour_bef_temperature,hour_bef_precipitation,hour_bef_windspeed,hour_bef_humidity,hour_bef_visibility,hour_bef_ozone,hour_bef_pm10,hour_bef_pm2.5
id,,,,,,,,,
0,7,20.7,0.0,1.3,62.0,954.0,0.041,44.0,27.0
1,17,30.0,0.0,5.4,33.0,1590.0,0.061,49.0,36.0
2,13,19.0,1.0,2.1,95.0,193.0,0.020,36.0,28.0
4,6,22.5,0.0,2.5,60.0,1185.0,0.027,52.0,38.0
5,22,14.6,1.0,3.4,93.0,218.0,0.041,18.0,15.0


,count
id,
0,NaN
1,NaN
2,NaN
4,NaN
5,NaN


In [4]:
display(train_csv.info())
display(test_csv.info())
display(submission.info())

<class 'pandas.DataFrame'>
Index: 1459 entries, 3 to 2179
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   hour                    1459 non-null   int64  
 1   hour_bef_temperature    1457 non-null   float64
 2   hour_bef_precipitation  1457 non-null   float64
 3   hour_bef_windspeed      1450 non-null   float64
 4   hour_bef_humidity       1457 non-null   float64
 5   hour_bef_visibility     1457 non-null   float64
 6   hour_bef_ozone          1383 non-null   float64
 7   hour_bef_pm10           1369 non-null   float64
 8   hour_bef_pm2.5          1342 non-null   float64
 9   count                   1459 non-null   float64
dtypes: float64(9), int64(1)
memory usage: 125.4 KB


None

<class 'pandas.DataFrame'>
Index: 715 entries, 0 to 2177
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   hour                    715 non-null    int64  
 1   hour_bef_temperature    714 non-null    float64
 2   hour_bef_precipitation  714 non-null    float64
 3   hour_bef_windspeed      714 non-null    float64
 4   hour_bef_humidity       714 non-null    float64
 5   hour_bef_visibility     714 non-null    float64
 6   hour_bef_ozone          680 non-null    float64
 7   hour_bef_pm10           678 non-null    float64
 8   hour_bef_pm2.5          679 non-null    float64
dtypes: float64(8), int64(1)
memory usage: 55.9 KB


None

<class 'pandas.DataFrame'>
Index: 715 entries, 0 to 2177
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   count   0 non-null      float64
dtypes: float64(1)
memory usage: 11.2 KB


None

In [ ]:
# train_csv를 x와 y로 분리
x = train_csv.drop(['count'], axis=1)   # 열(컬럼) 삭제
y = train_csv['count']

test_idx = test_csv.index
data = pd.concat([x, test_csv], axis=0)
display(data.sort_index(axis=0, ascending=True, inplace=False).head(10))
# 시계열 데이터 같아서 선형보간 해보려고했는데 시계열데이터는 아닌거같음

,hour,hour_bef_temperature,hour_bef_precipitation,hour_bef_windspeed,hour_bef_humidity,hour_bef_visibility,hour_bef_ozone,hour_bef_pm10,hour_bef_pm2.5
id,,,,,,,,,
0,7,20.7,0.0,1.3,62.0,954.0,0.041,44.0,27.0
1,17,30.0,0.0,5.4,33.0,1590.0,0.061,49.0,36.0
2,13,19.0,1.0,2.1,95.0,193.0,0.020,36.0,28.0
3,20,16.3,1.0,1.5,89.0,576.0,0.027,76.0,33.0
4,6,22.5,0.0,2.5,60.0,1185.0,0.027,52.0,38.0
5,22,14.6,1.0,3.4,93.0,218.0,0.041,18.0,15.0
6,13,20.1,0.0,1.4,48.0,916.0,0.042,73.0,40.0
7,6,13.9,0.0,0.7,79.0,1382.0,0.033,32.0,19.0
8,23,8.1,0.0,2.7,54.0,946.0,0.040,75.0,64.0
